
# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: fetch latest code</h2>
            <span style="color:#f71;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull and merge your changes as needed</a>. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/><br/>
            After you've pulled the code, from the llm_engineering directory, in a Cursor Terminal, run:<br/>
            <code>uv sync</code><br/>
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT-6.1 Sol, Claude Sonnet 5.5, Gemini 4 Argon, Grok 4.3, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-6-luna.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [6]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if groq_api_key:
    print(f"groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("groq API Key not set (and this is optional)")

OpenAI API Key not set
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AQ
groq API Key exists and begins gsk_


In [12]:
# Connect to client libraries

# openai = OpenAI()

# anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"

# anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [13]:
# OPENAI_MODEL = "gpt-6.1-sol"
# CLAUDE_MODEL = "claude-sonnet-5-5"
GROQ_MODEL = "openai/gpt-oss-120b"
GEMINI_MODEL = "gemini-3.8-flash"

# Want to keep costs ultra-low? Uncomment these lines:

# OPENAI_MODEL = "gpt-6-luna"
# CLAUDE_MODEL = "claude-haiku-4-5"
# GROK_MODEL = "grok-4-fast-non-reasoning"
# GEMINI_MODEL = "gemini-3.1-flash-lite"

## PLEASE NOTE:

We will be writing a solution to convert Python into efficient, optimized C++ code for your machine, which can be compiled to native machine code and executed.

It is not necessary for you to execute the code yourself - that's not the point of the exercise!

But if you would like to (because it's satisfying!) then I'm including the steps here. Very optional!

As an alternative, I'll also show you a website where you can run the C++ code.

In [14]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26300',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget'],
 'cpu': {'brand': '13th Gen Intel(R) Core(TM) i7-13700H',
  'cores_logical': 20,
  'cores_physical': 14,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [16]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so. 
Winget doesn't work, so please give me alternate way

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = groq.chat.completions.create(model=GROQ_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

**Short answer:**  
You do **not** have a C++ compiler installed yet (all entries under `toolchain → compilers` are empty).  
Below are two completely offline‑friendly ways to get a working compiler on Windows 11 without using `winget`. Pick the one you prefer, follow the step‑by‑step instructions, and then you can use the exact `compile_command` and `run_command` shown at the end of this answer inside your Python script.

---

## 1️⃣ Option A – Microsoft “Build Tools for Visual Studio” (MSVC)

### Why this is the “fastest possible runtime” on Windows
* MSVC’s optimizer (`/O2`) is the native optimizer that Microsoft ships with Windows, so the generated binaries are usually a bit faster than the same code built with GCC/Clang on the same machine.
* No extra runtime DLLs are required beyond the Windows CRT that is already present.

### Step‑by‑step installation (no winget, no internet‑dependent package manager)

| Step | Action |
|------|--------|
| **1.** | Open your web browser and go to the official download page: <br>**<https://visualstudio.microsoft.com/visual-cpp-build-tools/>** |
| **2.** | Click **“Download Build Tools”** – a small `vs_BuildTools.exe` (≈ 20 MB) will be saved. |
| **3.** | Run `vs_BuildTools.exe`. In the installer window **check** the workload **“Desktop development with C++”**. Under the *Individual components* tab also make sure that **“MSVC v143 – VS 2022 C++ x64/x86 tools”** and **“Windows 10 SDK (10.0.xxxxx.x)”** are selected. You can leave everything else at the default. |
| **4.** | Click **“Install”**. The installer will download the required files (≈ 2 GB) and complete the installation. This may take a few minutes. |
| **5.** | When the installation finishes, close the installer. |
| **6.** | **Add the compiler to your PATH** (so you can call `cl.exe` from any command‑prompt). The easiest way is to use the *Developer Command Prompt* that the installer creates, but you can also permanently add the path:  <br>Open **Start → Settings → Environment Variables → Edit the system PATH** and add: <br>```\`C:\Program Files\Microsoft Visual Studio\2022\BuildTools\VC\Tools\MSVC\<version>\bin\Hostx64\x64```  <br>(Replace `<version>` with the folder you see, e.g. `14.38.33130`). |
| **7.** | Open a **regular** Command Prompt (or PowerShell) and run `cl` – you should see the Microsoft C/C++ compiler version printed. If you get “`'cl' is not recognized`”, double‑check the PATH entry you added. |

> **Tip:** If you only ever run the compiler from Python, you can also invoke the *Developer Command Prompt* batch file (`vcvarsall.bat`) from Python before calling `cl`. That way you don’t need to modify the system PATH at all (see the “Python snippet” below).

---

## 2️⃣ Option B – MinGW‑w64 (GCC/Clang) – pure “stand‑alone” binary

If you prefer the GNU tool‑chain, this method downloads a pre‑built MinGW‑w64 package and puts it on your PATH.

### Step‑by‑step installation (no winget)

| Step | Action |
|------|--------|
| **1.** | Download the latest **MinGW‑w64** “win‑libs” zip from the official source: <br>**<https://github.com/niXman/mingw-builds/releases/latest>** (look for a file named something like `x86_64-posix-seh-rev0.13.0-rt_v12-rev0.13.0-mingw-w64-v12.2.0.zip`). |
| **2.** | Extract the zip to a folder of your choice, e.g. `C:\mingw-w64`. The folder will contain `bin`, `include`, `lib`, … |
| **3.** | Add the `bin` directory to your system PATH: <br>Open **Start → Settings → Environment Variables → Edit the system PATH** and add `C:\mingw-w64\bin`. |
| **4.** | Open a new Command Prompt and type `g++ --version`. You should see something like `g++ (x86_64-posix-seh-rev0, Built by ...)`. |
| **5.** | (Optional) If you want **Clang** that comes bundled with the same package, the executable is `clang++` in the same `bin` folder. |

> **Note:** GCC’s optimizer (`-O2`/`-O3`) is very good, but on Windows the MSVC‑generated binaries are usually a tad faster for pure‑CPU code because they can make use of the newest Windows‑only intrinsics.

---

## 3️⃣ Using the compiler from Python

Below are **exact** `compile_command` and `run_command` lists you can paste into your script.  
Both examples assume the source file is `main.cpp` in the **current working directory**.

### 3.1 When you installed **MSVC** (option A)

```python
import subprocess, os, sys

# ----------------------------------------------------------------------
# 1) Ensure the MSVC environment variables are set.
# ----------------------------------------------------------------------
# The easiest way is to run the vcvarsall.bat script first.
# It lives at:
#   C:\Program Files\Microsoft Visual Studio\2022\BuildTools\VC\Auxiliary\Build\vcvarsall.bat
# (adjust the path if you installed VS in a different location.)

vcvars = r"C:\Program Files\Microsoft Visual Studio\2022\BuildTools\VC\Auxiliary\Build\vcvarsall.bat"
# The argument "x64" tells it to set up a 64‑bit host+target environment.
cmd = [vcvars, "x64", "&&", "cl", "/nologo", "/O2", "main.cpp", "/Fe:main.exe"]
compile_command = ["cmd.exe", "/c"] + cmd   # run through cmd so the && works

# ----------------------------------------------------------------------
# 2) Run the compiled program
# ----------------------------------------------------------------------
run_command = ["cmd.exe", "/c", "main.exe"]   # works from any folder

# ----------------------------------------------------------------------
# 3) Execute
# ----------------------------------------------------------------------
compile_result = subprocess.run(
    compile_command,
    check=True,
    text=True,
    capture_output=True,
)

run_result = subprocess.run(
    run_command,
    check=True,
    text=True,
    capture_output=True,
)

print("Compiler output:", compile_result.stdout)
print("Program output :", run_result.stdout)
```

**Explanation of the flags**

| Flag | Meaning |
|------|----------|
| `/nologo` | Suppress the startup banner – makes the captured output cleaner. |
| `/O2` | Full speed optimization (equivalent to `-O2` in GCC). |
| `main.cpp` | Your source file. |
| `/Fe:main.exe` | Name of the produced executable. |

If you **already added the MSVC `bin` folder to PATH**, you can drop the `vcvarsall.bat` step and simply use:

```python
compile_command = ["cl", "/nologo", "/O2", "main.cpp", "/Fe:main.exe"]
run_command = ["main.exe"]
```

### 3.2 When you installed **MinGW‑w64** (option B)

```python
import subprocess

# ----------------------------------------------------------------------
# 1) Compile with g++
# ----------------------------------------------------------------------
compile_command = [
    "g++",                     # the compiler executable (must be on PATH)
    "-std=c++20",              # modern language version (adjust if you like)
    "-O2",                     # speed‑optimisation
    "-march=native",           # generate code tuned for this CPU
    "main.cpp",
    "-o", "main.exe",          # output file name
]

# ----------------------------------------------------------------------
# 2) Run the program
# ----------------------------------------------------------------------
run_command = ["./main.exe"]   # on Windows "./" works, or just "main.exe"

# ----------------------------------------------------------------------
# 3) Execute
# ----------------------------------------------------------------------
compile_result = subprocess.run(
    compile_command,
    check=True,
    text=True,
    capture_output=True,
)

run_result = subprocess.run(
    run_command,
    check=True,
    text=True,
    capture_output=True,
)

print("Compiler output:", compile_result.stdout)
print("Program output :", run_result.stdout)
```

**Explanation of the flags**

| Flag | Meaning |
|------|----------|
| `-std=c++20` | Use the latest language features (you can downgrade to `c++17` if you prefer). |
| `-O2` | Optimize for speed (good compromise between compile time and runtime). |
| `-march=native` | Let GCC emit instructions that your CPU (13th‑gen Intel i7‑13700H) actually supports (e.g., AVX‑512). |
| `-o main.exe` | Output executable name. |

If you want the **absolute fastest** you could replace `-O2` with `-O3 -ffast-math -funroll-loops`, but note that those flags can change floating‑point semantics.

---

## 4️⃣ Verifying the whole pipeline

1. **Create a tiny test program** (`main.cpp`) in the same folder as your Python script:

   ```cpp
   #include <iostream>
   int main() {
       std::cout << "Hello from C++!\n";
       return 0;
   }
   ```

2. **Run your Python script** (`python compile_and_run.py`).  
   You should see something like:

   ```
   Compiler output:
   ... (compiler diagnostics, if any) ...

   Program output :
   Hello from C++!
   ```

If you see the expected “Hello from C++!” line, the compiler is correctly installed and the Python `subprocess` commands are working.

---

## 5️⃣ Quick‑look at “do I need to install anything?”

* **Yes** – your system reports *no* C++ compilers (`gcc`, `g++`, `clang`, `msvc_cl` are all empty).  
* Install **either** the Microsoft Build Tools **or** MinGW‑w64 using the steps above. Both are free, official binaries and do not rely on `winget`.

---

### TL;DR Cheat‑sheet

| Desired tool | Download link | Core install steps | Compile command (Python) | Run command (Python) |
|--------------|---------------|--------------------|--------------------------|----------------------|
| **MSVC** (fastest native) | <https://visualstudio.microsoft.com/visual-cpp-build-tools/> | Run installer → “Desktop development with C++” → add `…\MSVC\<ver>\bin\Hostx64\x64` to PATH (or call `vcvarsall.bat`) | `["cl", "/nologo", "/O2", "main.cpp", "/Fe:main.exe"]` | `["main.exe"]` |
| **MinGW‑w64** (GCC/Clang) | <https://github.com/niXman/mingw-builds/releases/latest> | Extract zip → add `C:\mingw-w64\bin` to PATH | `["g++", "-std=c++20", "-O2", "-march=native", "main.cpp", "-o", "main.exe"]` | `["./main.exe"]` |

Pick the row that matches the tool you installed, paste the corresponding two command lists into your Python code, and you’re ready to compile and run C++ from Python in the simplest possible way. Happy coding!

## If you need to install something

If you would like to, please follow GPTs instructions! Then rerun the analysis afterwards (you might need to Restart the notebook) to confirm you're set.

You should now be equipped with the command to compile the code, and the command to run it!

Enter that in the cell below:

In [ ]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

## And now, on with the main task

In [17]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [18]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [19]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [21]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [22]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [23]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [24]:
run_python(pi)

Result: 3.141592656089
Execution Time: 36.009420 seconds


In [25]:
port(groq, GROQ_MODEL, pi)

NameError: name 'compile_command' is not defined

# Compiling C++ and executing

This next cell contains the command to compile a C++ file based on the instructions from GPT.

Again, it's not crucial to do this step if you don't wish to!

OR alternatively: student Sandeep K.G. points out that you can run Python and C++ code online to test it out that way. Thank you Sandeep!  
> Not an exact comparison but you can still get the idea of performance difference.  
> For example here: https://www.programiz.com/cpp-programming/online-compiler/

In [ ]:
# Use the commands from GPT-6.1 Sol

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [ ]:
compile_and_run()

In [ ]:
19.178207/0.082168

## OK let's try the other contenders!

In [ ]:
port(anthropic, CLAUDE_MODEL, pi)
compile_and_run()

In [ ]:
port(grok, GROK_MODEL, pi)
compile_and_run()

In [ ]:
port(gemini, GEMINI_MODEL, pi)
compile_and_run()


In [ ]:
print(f"""
In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: {19.178207/0.104241:.0f}X speedup
3rd place: GPT-5: {19.178207/0.082168:.0f}X speedup
2nd place: Grok 4: {19.178207/0.018092:.0f}X speedup
1st place: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup
""")